# Study 911 — REIT Quality Screen 🏢

**Not all REITs are equal — does a "quality" screen beat the broad index?**

Equity REITs own property and collect durable rents at moderate leverage; **mortgage REITs**
lever a thin spread between long mortgage assets and short funding and pay it out as a fat,
fragile dividend. The "quality REIT" screen holds the durable-income equity sleeve
(residential **REZ**, broad **VNQ**/**RWR**) and screens *out* the levered-carry sleeve
(**REM**), aiming to beat the broad index on a **risk-adjusted, net-of-cost** basis.

We race the live vehicles excess-of-BIL over **2007-06 → 2026-06** (229
months). *Numbers below are the frozen headline (`docs/results.md`); the live cells run the
fast synthetic control. These sector ETFs are young — magnitudes are indicative.*


## 1. The race in one table

Excess-vs-excess Sharpe (every sleeve minus the BIL T-bill), 2007–2026. Watch the residential quality sleeve (REZ) edge the broad index (VNQ) — and watch the mortgage-REIT sleeve (REM) earn a **negative** 19-year total return despite its famous fat dividend.

In [1]:
R = {'start': '2007-06', 'end': '2026-06', 'n_months': 229, 'fingerprint': 'b99a4946b405', 'sh_rez': 0.36, 'sh_vnq': 0.29, 'sh_rwr': 0.27, 'sh_rem': 0.039, 'sh_spy': 0.644, 'ann_rez': 6.83, 'ann_vnq': 5.4, 'ann_rwr': 4.95, 'ann_rem': -1.13, 'ann_spy': 10.68, 'vol_rez': 20.9, 'vol_vnq': 22.3, 'vol_rem': 24.2, 'rezvnq_bps': 8.7, 'rezvnq_t': 0.75, 'book_bps': 2.0, 'book_t': 0.41, 'adv': 0.07, 'adv_lo': -0.07, 'adv_hi': 0.202, 'adv_fracneg': 0.17, 'era1_rezvnq_bps': 1.4, 'era1_rezvnq_t': 0.08, 'era2_rezvnq_bps': 16.2, 'era2_rezvnq_t': 1.09, 'rezrem_bps': 55.0, 'rezrem_t': 1.72, 'era1_rezrem_bps': 92.7, 'era1_rezrem_t': 1.83, 'era2_rezrem_bps': 17.1, 'era2_rezrem_t': 0.47, 'dd_rez': -66.9, 'dd_vnq': -73.1, 'dd_rem': -74.7, 'dd_spy': -55.2, 'cost2_net': 1.8, 'cost2_t': 0.37, 'cost2_ann': 0.22, 'cost5_net': 1.5, 'cost5_t': 0.31, 'cost5_ann': 0.18, 'cost10_net': 1.0, 'cost10_t': 0.21, 'cost10_ann': 0.12, 'null_adv_mean': -0.019, 'null_straddle': 17, 'null_trapflag': 20, 'planted_t': 3.17, 'planted_adv': 0.109, 'planted_adv_lo': 0.04}

In [2]:
print('sleeve         excessSharpe   ann.total-return')
print(f"REZ (quality)  {R['sh_rez']:+.3f}        {R['ann_rez']:+.2f}%/yr")
print(f"VNQ (broad)    {R['sh_vnq']:+.3f}        {R['ann_vnq']:+.2f}%/yr")
print(f"REM (mREIT)    {R['sh_rem']:+.3f}        {R['ann_rem']:+.2f}%/yr   <- the levered-carry trap")

sleeve         excessSharpe   ann.total-return
REZ (quality)  +0.360        +6.83%/yr
VNQ (broad)    +0.290        +5.40%/yr
REM (mREIT)    +0.039        -1.13%/yr   <- the levered-carry trap


## 2. Is the machinery honest? A live synthetic control

We plant a quality edge in a seeded toy world (`edge>0`) and check the Sharpe-advantage estimator recovers it — and that it stays centred at zero on the null (`edge=0`), while the trap detector always flags the inferior-Sharpe levered leg. No network.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from reit_quality import data, strategy as st
null = st.sharpe_advantage(data.synthetic_world(edge_ann=0.0, seed=911), 'QUAL','BROAD', rf='CASH', n_boot=600)
plant = st.sharpe_advantage(data.synthetic_world(edge_ann=0.03, seed=911), 'QUAL','BROAD', rf='CASH', n_boot=600)
print('null   : Sharpe adv %+.3f  CI [%+.3f, %+.3f]  (straddles 0)' % (null['advantage'], null['ci_low'], null['ci_high']))
print('planted: Sharpe adv %+.3f  CI [%+.3f, %+.3f]  (clear of 0)' % (plant['advantage'], plant['ci_low'], plant['ci_high']))

null   : Sharpe adv -0.046  CI [-0.121, +0.031]  (straddles 0)
planted: Sharpe adv +0.109  CI [+0.040, +0.187]  (clear of 0)


## 3. The honest verdict — one real distinction, no bankable edge

**The durable-income tilt is *not* certified.** REZ edges VNQ on Sharpe (0.36 vs 0.29), but the monthly spread is only **+8.7 bps/mo at HAC *t* = 0.75**, the bootstrap Sharpe-advantage CI **[-0.070, +0.202] straddles zero**, and all of the (insignificant) tilt lives in the second half (t = 0.08 → 1.09).

**The levered-carry *trap* is real and structural.** Mortgage REITs earned **-1.13%/yr for 19 years** at excess Sharpe 0.039 — an order of magnitude below the equity sleeves. *But* the broad index VNQ already holds ≈ no mortgage REITs, so avoiding the trap is **already free**.

**Signal: Mixed** (tilt absent, trap real) · **Tradability: Fragile** (the one robust action is already inside the index you'd otherwise hold; the incremental tilt nets ~+0.1–0.2%/yr at *t* ≈ 0.3).